In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import networkx as nx
from shapely.geometry import LineString, MultiLineString

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import networkx as nx
from shapely.geometry import LineString, MultiLineString

# =========================================================
# 1. PARAMÈTRES
# =========================================================

SHP_PATH = "ROUTE500_3-0__SHP_LAMB93_FXX_2021-11-03/ROUTE500/1_DONNEES_LIVRAISON_2022-01-00175/R500_3-0_SHP_LAMB93_FXX-ED211/RESEAU_ROUTIER/TRONCON_ROUTE.shp"   # à adapter
USE_LONGUEUR_COLUMN = True  # utiliser LONGUEUR si elle est propre

# =========================================================
# 2. CHARGEMENT
# =========================================================

gdf = gpd.read_file(SHP_PATH, encoding="latin1")

print("Colonnes disponibles :")
print(gdf.columns.tolist())
print("\nCRS initial :", gdf.crs)
print("Nombre de lignes initial :", len(gdf))

# Reprojection en Lambert-93 si besoin
if gdf.crs is None or gdf.crs.to_epsg() != 2154:
    gdf = gdf.to_crs(2154)

print("CRS après reprojection :", gdf.crs)

# =========================================================
# 3. NETTOYAGE
# =========================================================

gdf = gdf[gdf.geometry.notna()].copy()
gdf = gdf[gdf.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()

# Longueur
if USE_LONGUEUR_COLUMN and "LONGUEUR" in gdf.columns:
    gdf["LONGUEUR"] = pd.to_numeric(gdf["LONGUEUR"], errors="coerce")
    gdf["length_m"] = gdf["LONGUEUR"].fillna(gdf.geometry.length)
else:
    gdf["length_m"] = gdf.geometry.length

gdf = gdf[gdf["length_m"] > 0].copy()

print("\nNombre de lignes après nettoyage :", len(gdf))

# =========================================================
# 4. DIAGNOSTIC RAPIDE DES MODALITÉS
# =========================================================

cols_to_check = ["VOCATION", "CLASS_ADM", "ETAT", "ACCES", "NB_CHAUSSE", "NB_VOIES", "SENS"]
for col in cols_to_check:
    if col in gdf.columns:
        print(f"\n--- {col} ---")
        print(gdf[col].astype(str).value_counts(dropna=False).head(20))

# =========================================================
# 5. FONCTION D'ESTIMATION DE VITESSE
# =========================================================

def estimate_speed(row):
    """
    Estimation heuristique de vitesse en km/h.
    À ajuster selon les valeurs exactes observées dans ton dataset.
    """

    speed = 80  # base par défaut

    vocation = str(row["VOCATION"]).strip().lower() if "VOCATION" in row and pd.notna(row["VOCATION"]) else ""
    class_adm = str(row["CLASS_ADM"]).strip().lower() if "CLASS_ADM" in row and pd.notna(row["CLASS_ADM"]) else ""
    etat = str(row["ETAT"]).strip().lower() if "ETAT" in row and pd.notna(row["ETAT"]) else ""
    acces = str(row["ACCES"]).strip().lower() if "ACCES" in row and pd.notna(row["ACCES"]) else ""
    nb_chausse = str(row["NB_CHAUSSE"]).strip().lower() if "NB_CHAUSSE" in row and pd.notna(row["NB_CHAUSSE"]) else ""
    nb_voies = str(row["NB_VOIES"]).strip().lower() if "NB_VOIES" in row and pd.notna(row["NB_VOIES"]) else ""

    # 1) Vocation / type implicite de route
    if "liaison principale" in vocation:
        speed = 110
    elif "liaison régionale" in vocation:
        speed = 90
    elif "liaison locale" in vocation:
        speed = 70
    elif "bretelle" in vocation:
        speed = 50

    # 2) Classe administrative
    if "autoroute" in class_adm:
        speed = max(speed, 120)
    elif "nationale" in class_adm:
        speed = max(speed, 90)
    elif "départementale" in class_adm or "departementale" in class_adm:
        speed = max(speed, 80)
    elif "communale" in class_adm:
        speed = min(speed, 50)

    # 3) Nombre de chaussées
    if "2 chauss" in nb_chausse or "double" in nb_chausse:
        speed += 10
    elif "1 chauss" in nb_chausse or "simple" in nb_chausse:
        speed += 0

    # 4) Nombre de voies
    # si la variable est textuelle et parfois exploitable
    if "4" in nb_voies or "3" in nb_voies:
        speed += 10
    elif "1" in nb_voies:
        speed -= 10

    # 5) Etat / accès
    if "en construction" in etat or "projet" in etat:
        speed = 30

    if "interdit" in acces or "fermé" in acces or "ferme" in acces:
        speed = 5

    # bornes de sécurité
    speed = max(5, min(speed, 130))
    return speed

gdf["speed_kmh"] = gdf.apply(estimate_speed, axis=1)

# Temps de parcours en secondes
gdf["travel_time_s"] = gdf["length_m"] / (gdf["speed_kmh"] * 1000 / 3600)

print("\nAperçu longueur / vitesse / temps :")
print(gdf[["length_m", "speed_kmh", "travel_time_s"]].head(10).to_string())

# =========================================================
# 6. EXTRACTION DES EXTRÉMITÉS
# =========================================================

def get_endpoints(geom):
    if geom is None:
        return None, None

    if isinstance(geom, LineString):
        coords = list(geom.coords)
        if len(coords) < 2:
            return None, None
        return tuple(coords[0]), tuple(coords[-1])

    if isinstance(geom, MultiLineString):
        parts = list(geom.geoms)
        if len(parts) == 0:
            return None, None
        first_coords = list(parts[0].coords)
        last_coords = list(parts[-1].coords)
        if len(first_coords) == 0 or len(last_coords) == 0:
            return None, None
        return tuple(first_coords[0]), tuple(last_coords[-1])

    return None, None

gdf[["u", "v"]] = gdf.apply(lambda row: pd.Series(get_endpoints(row.geometry)), axis=1)
gdf = gdf[gdf["u"].notna() & gdf["v"].notna()].copy()

print("\nNombre de tronçons exploitables :", len(gdf))

# =========================================================
# 7. CONSTRUCTION DU GRAPHE ROUTIER
# =========================================================

G = nx.DiGraph()

def normalize_sens(val):
    if pd.isna(val):
        return "double sens"
    return str(val).strip().lower()

for _, row in gdf.iterrows():
    u = row["u"]
    v = row["v"]

    attrs = {
        "id_rte500": row["ID_RTE500"] if "ID_RTE500" in gdf.columns else None,
        "length_m": float(row["length_m"]),
        "speed_kmh": float(row["speed_kmh"]),
        "travel_time_s": float(row["travel_time_s"]),
        "vocation": row["VOCATION"] if "VOCATION" in gdf.columns else None,
        "nb_chausse": row["NB_CHAUSSE"] if "NB_CHAUSSE" in gdf.columns else None,
        "nb_voies": row["NB_VOIES"] if "NB_VOIES" in gdf.columns else None,
        "etat": row["ETAT"] if "ETAT" in gdf.columns else None,
        "acces": row["ACCES"] if "ACCES" in gdf.columns else None,
        "num_route": row["NUM_ROUTE"] if "NUM_ROUTE" in gdf.columns else None,
        "class_adm": row["CLASS_ADM"] if "CLASS_ADM" in gdf.columns else None,
        "geometry": row["geometry"]
    }

    sens = normalize_sens(row["SENS"]) if "SENS" in gdf.columns else "double sens"

    if "inverse" in sens:
        G.add_edge(v, u, **attrs)
    elif "direct" in sens:
        G.add_edge(u, v, **attrs)
    else:
        G.add_edge(u, v, **attrs)
        G.add_edge(v, u, **attrs)

print("\nGraphe construit :")
print("Nombre de nœuds :", G.number_of_nodes())
print("Nombre d’arcs :", G.number_of_edges())

# =========================================================
# 8. FONCTION NOEUD LE PLUS PROCHE
# =========================================================

nodes_list = list(G.nodes())
nodes_array = np.array(nodes_list)

def nearest_node(x, y):
    diffs = nodes_array - np.array([x, y])
    dists = np.sqrt((diffs ** 2).sum(axis=1))
    idx = np.argmin(dists)
    return tuple(nodes_array[idx])

# =========================================================
# 9. EXEMPLE DE ROUTAGE
# =========================================================

sample = gdf.sample(2, random_state=42)

origin_geom = sample.iloc[0].geometry
dest_geom = sample.iloc[1].geometry

origin_point = origin_geom.interpolate(0.5, normalized=True)
dest_point = dest_geom.interpolate(0.5, normalized=True)

origin_node = nearest_node(origin_point.x, origin_point.y)
dest_node = nearest_node(dest_point.x, dest_point.y)

print("\nOrigine :", origin_node)
print("Destination :", dest_node)

try:
    path = nx.shortest_path(G, source=origin_node, target=dest_node, weight="travel_time_s")
    travel_time_total_s = nx.shortest_path_length(G, source=origin_node, target=dest_node, weight="travel_time_s")

    total_length_m = 0
    for i in range(len(path) - 1):
        total_length_m += G[path[i]][path[i + 1]]["length_m"]

    print("\nChemin trouvé :")
    print("Nombre de nœuds :", len(path))
    print("Distance totale (km) :", round(total_length_m / 1000, 2))
    print("Temps estimé (min) :", round(travel_time_total_s / 60, 2))

except nx.NetworkXNoPath:
    print("\nAucun chemin trouvé entre ces deux nœuds.")
except Exception as e:
    print("\nErreur :", e)

# =========================================================
# 10. EXPORT TABLE PROPRE
# =========================================================

export_cols = [
    "ID_RTE500", "VOCATION", "NB_CHAUSSE", "NB_VOIES", "ETAT", "ACCES",
    "SENS", "NUM_ROUTE", "LONGUEUR", "CLASS_ADM",
    "length_m", "speed_kmh", "travel_time_s", "u", "v", "geometry"
]

export_cols = [c for c in export_cols if c in gdf.columns]
gdf_export = gdf[export_cols].copy()

print("\nTable export :")
print(gdf_export.head(10).to_string())

# gdf_export.to_file("route500_cleaned.gpkg", driver="GPKG")
# gdf_export.to_parquet("route500_cleaned.parquet")

Colonnes disponibles :
['ID_RTE500', 'VOCATION', 'NB_CHAUSSE', 'NB_VOIES', 'ETAT', 'ACCES', 'RES_VERT', 'SENS', 'NUM_ROUTE', 'RES_EUROPE', 'LONGUEUR', 'CLASS_ADM', 'geometry']

CRS initial : PROJCS["RGF93 Lambert 93",GEOGCS["RGF93 geographiques (dms)",DATUM["Reseau_Geodesique_Francais_1993_v1",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6171"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["IGNF","RGF93G"]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",46.5],PARAMETER["central_meridian",3],PARAMETER["standard_parallel_1",44],PARAMETER["standard_parallel_2",49],PARAMETER["false_easting",700000],PARAMETER["false_northing",6600000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["IGNF","LAMB93"]]
Nombre de lignes initial : 1302758
CRS après reprojection : PROJCS["RGF93 Lambert 93",GEOGCS["RGF93 geographiqu

In [3]:
for col in ["VOCATION", "CLASS_ADM", "ETAT", "ACCES", "NB_CHAUSSE", "SENS"]:
    print(f"\n### {col}")
    print(gdf[col].astype(str).value_counts(dropna=False).head(30))


### VOCATION
VOCATION
Liaison locale        975778
Liaison régionale     213508
Liaison principale     98562
Type autoroutier       14777
Bretelle                 131
Name: count, dtype: int64

### CLASS_ADM
CLASS_ADM
Sans objet        713365
Départementale    568099
Nationale          13535
Autoroute           7757
Name: count, dtype: int64

### ETAT
ETAT
Revêtu    1302756
Name: count, dtype: int64

### ACCES
ACCES
Libre      1299266
A péage       3490
Name: count, dtype: int64

### NB_CHAUSSE
NB_CHAUSSE
1 chaussée     1273709
2 chaussées      29047
Name: count, dtype: int64

### SENS
SENS
Double sens     1292813
Sens direct        8127
Sens inverse       1816
Name: count, dtype: int64


In [4]:
gdf["hour"] = np.random.choice(range(24), size=len(gdf))
gdf["is_peak"] = gdf["hour"].isin([7, 8, 9, 17, 18, 19]).astype(int)

gdf["congestion_factor"] = np.where(
    gdf["is_peak"] == 1,
    np.random.uniform(1.2, 1.8, size=len(gdf)),
    np.random.uniform(0.9, 1.1, size=len(gdf))
)

gdf["travel_time_congested_s"] = gdf["travel_time_s"] * gdf["congestion_factor"]


In [5]:
gdf_work = gdf.copy()

# 1) Longueur
if "length_m" not in gdf_work.columns:
    if "LONGUEUR" in gdf_work.columns:
        gdf_work["LONGUEUR"] = pd.to_numeric(gdf_work["LONGUEUR"], errors="coerce")
        gdf_work["length_m"] = gdf_work["LONGUEUR"].fillna(gdf_work.geometry.length)
    else:
        gdf_work["length_m"] = gdf_work.geometry.length

gdf_work = gdf_work[gdf_work.geometry.notna()].copy()
gdf_work = gdf_work[gdf_work.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()
gdf_work = gdf_work[gdf_work["length_m"] > 0].copy()

# 2) Vitesse heuristique adaptée à ton dataset
VOCATION_SPEED = {
    "Type autoroutier": 130,
    "Bretelle": 50,
    "Liaison principale": 100,
    "Liaison régionale": 80,
    "Liaison locale": 60
}

CLASS_ADM_ADJ = {
    "Autoroute": 10,
    "Nationale": 5,
    "Départementale": 0,
    "Sans objet": 0
}

NB_CHAUSSE_ADJ = {
    "2 chaussées": 10,
    "1 chaussée": 0
}

ACCES_ADJ = {
    "A péage": 5,
    "Libre": 0
}

def estimate_speed(row):
    speed = VOCATION_SPEED.get(row.get("VOCATION"), 70)
    speed += CLASS_ADM_ADJ.get(row.get("CLASS_ADM"), 0)
    speed += NB_CHAUSSE_ADJ.get(row.get("NB_CHAUSSE"), 0)
    speed += ACCES_ADJ.get(row.get("ACCES"), 0)
    return max(30, min(speed, 130))

gdf_work["speed_kmh"] = gdf_work.apply(estimate_speed, axis=1)
gdf_work["travel_time_base_s"] = gdf_work["length_m"] / (gdf_work["speed_kmh"] * 1000 / 3600)

# 3) Extraction des extrémités
def get_endpoints(geom):
    if geom is None:
        return None, None

    if geom.geom_type == "LineString":
        coords = list(geom.coords)
        if len(coords) < 2:
            return None, None
        return tuple(coords[0]), tuple(coords[-1])

    if geom.geom_type == "MultiLineString":
        parts = list(geom.geoms)
        if not parts:
            return None, None
        first_coords = list(parts[0].coords)
        last_coords = list(parts[-1].coords)
        if not first_coords or not last_coords:
            return None, None
        return tuple(first_coords[0]), tuple(last_coords[-1])

    return None, None

gdf_work[["u", "v"]] = gdf_work.apply(
    lambda row: pd.Series(get_endpoints(row.geometry)),
    axis=1
)

gdf_work = gdf_work[gdf_work["u"].notna() & gdf_work["v"].notna()].copy()

# 4) Construction du graphe orienté
G_base = nx.DiGraph()

def normalize_sens(val):
    if pd.isna(val):
        return "double sens"
    return str(val).strip().lower()

for _, row in gdf_work.iterrows():
    u, v = row["u"], row["v"]

    attrs = {
        "length_m": float(row["length_m"]),
        "speed_kmh": float(row["speed_kmh"]),
        "travel_time_base_s": float(row["travel_time_base_s"]),
        "vocation": row.get("VOCATION"),
        "class_adm": row.get("CLASS_ADM"),
        "nb_chausse": row.get("NB_CHAUSSE"),
        "acces": row.get("ACCES"),
        "geometry": row.geometry
    }

    sens = normalize_sens(row.get("SENS"))

    if "inverse" in sens:
        G_base.add_edge(v, u, **attrs)
    elif "direct" in sens:
        G_base.add_edge(u, v, **attrs)
    else:
        G_base.add_edge(u, v, **attrs)
        G_base.add_edge(v, u, **attrs)

print("gdf_work prêt :", gdf_work.shape)
print("Nb nœuds :", G_base.number_of_nodes())
print("Nb arcs :", G_base.number_of_edges())

display(
    gdf_work[
        ["VOCATION", "CLASS_ADM", "NB_CHAUSSE", "ACCES", "length_m", "speed_kmh", "travel_time_base_s", "u", "v"]
    ].head(10)
)


gdf_work prêt : (1302756, 23)
Nb nœuds : 911769
Nb arcs : 2575411


,VOCATION,CLASS_ADM,NB_CHAUSSE,ACCES,length_m,speed_kmh,travel_time_base_s,u,v
0,Type autoroutier,Autoroute,2 chaussées,Libre,3.18,130,0.088062,"(894833.5, 6265743.5)","(894051.2, 6262805.5)"
1,Type autoroutier,Autoroute,2 chaussées,Libre,1.29,130,0.035723,"(894051.2, 6262805.5)","(893427.9, 6261750.6)"
2,Type autoroutier,Autoroute,2 chaussées,A péage,0.32,130,0.008862,"(340196.8, 6276241.3)","(340108.9, 6275932.5)"
3,Type autoroutier,Sans objet,2 chaussées,Libre,0.25,130,0.006923,"(433102.7, 6715831.8)","(433282.0, 6716003.9)"
4,Type autoroutier,Sans objet,2 chaussées,Libre,0.11,130,0.003046,"(433378.5, 6716062.7)","(433282.0, 6716003.9)"
5,Type autoroutier,Départementale,2 chaussées,Libre,0.25,130,0.006923,"(432966.6, 6715628.5)","(433102.7, 6715831.8)"
6,Type autoroutier,Nationale,2 chaussées,Libre,6.37,130,0.176400,"(789673.0, 6595342.3)","(795802.6, 6594100.4)"
7,Type autoroutier,Autoroute,2 chaussées,A péage,0.63,130,0.017446,"(838860.6, 6576862.1)","(839472.1, 6576732.0)"
8,Type autoroutier,Autoroute,2 chaussées,A péage,0.30,130,0.008308,"(512363.7, 6778175.0)","(512084.2, 6778080.2)"
9,Type autoroutier,Autoroute,2 chaussées,A péage,0.43,130,0.011908,"(511958.9, 6778047.5)","(511533.7, 6777981.0)"


In [6]:
# === Test rapide d'itinéraire sur G_base ===

nodes_list = list(G_base.nodes())
nodes_array = np.array(nodes_list)

def nearest_node(x, y):
    diffs = nodes_array - np.array([x, y])
    dists = np.sqrt((diffs ** 2).sum(axis=1))
    idx = np.argmin(dists)
    return tuple(nodes_array[idx])

sample = gdf_work.sample(2, random_state=42)

origin_geom = sample.iloc[0].geometry
dest_geom = sample.iloc[1].geometry

origin_point = origin_geom.interpolate(0.5, normalized=True)
dest_point = dest_geom.interpolate(0.5, normalized=True)

origin_node = nearest_node(origin_point.x, origin_point.y)
dest_node = nearest_node(dest_point.x, dest_point.y)

try:
    path = nx.shortest_path(G_base, source=origin_node, target=dest_node, weight="travel_time_base_s")
    time_s = nx.shortest_path_length(G_base, source=origin_node, target=dest_node, weight="travel_time_base_s")
    dist_m = sum(G_base[path[i]][path[i+1]]["length_m"] for i in range(len(path)-1))

    print("Origine :", origin_node)
    print("Destination :", dest_node)
    print("Nb nœuds du chemin :", len(path))
    print("Distance totale (km) :", round(dist_m / 1000, 2))
    print("Temps estimé (min) :", round(time_s / 60, 2))

except nx.NetworkXNoPath:
    print("Aucun chemin trouvé entre ces deux nœuds.")

Origine : (np.float64(563662.9), np.float64(6522773.2))
Destination : (np.float64(536199.2), np.float64(6688370.7))
Nb nœuds du chemin : 234
Distance totale (km) : 0.21
Temps estimé (min) : 0.11


In [7]:
# === Cellule 1 : congestion horaire simulée + OD ===

import folium
from shapely.geometry import Point

# Recrée nearest_node si besoin
nodes_list = list(G_base.nodes())
nodes_array = np.array(nodes_list)

def nearest_node(x, y):
    diffs = nodes_array - np.array([x, y])
    dists = np.sqrt((diffs ** 2).sum(axis=1))
    idx = np.argmin(dists)
    return tuple(nodes_array[idx])

def congestion_factor(attrs, hour):
    """
    Facteur multiplicatif simulé appliqué au temps de base.
    > 1 = plus lent (congestion)
    < 1 = plus fluide
    """
    vocation = attrs.get("vocation", "Liaison locale")
    nb_chausse = attrs.get("nb_chausse", "1 chaussée")
    acces = attrs.get("acces", "Libre")

    # Base par heure
    if hour in [7, 8, 9, 17, 18, 19]:  # pointe
        factor_map = {
            "Liaison locale": 1.45,
            "Liaison régionale": 1.28,
            "Liaison principale": 1.20,
            "Type autoroutier": 1.15,
            "Bretelle": 1.35
        }
    elif hour in [6, 10, 16, 20]:  # transition
        factor_map = {
            "Liaison locale": 1.18,
            "Liaison régionale": 1.10,
            "Liaison principale": 1.07,
            "Type autoroutier": 1.05,
            "Bretelle": 1.15
        }
    elif hour in [11, 12, 13, 14, 15]:  # journée
        factor_map = {
            "Liaison locale": 1.08,
            "Liaison régionale": 1.04,
            "Liaison principale": 1.02,
            "Type autoroutier": 1.01,
            "Bretelle": 1.08
        }
    else:  # nuit
        factor_map = {
            "Liaison locale": 0.92,
            "Liaison régionale": 0.95,
            "Liaison principale": 0.97,
            "Type autoroutier": 0.98,
            "Bretelle": 1.00
        }

    factor = factor_map.get(vocation, 1.0)

    # Ajustements structurels
    if nb_chausse == "2 chaussées":
        factor *= 0.95

    if acces == "A péage":
        factor *= 0.97

    return max(0.85, min(factor, 1.80))

def weight_for_hour(hour):
    def _weight(u, v, d):
        return d["travel_time_base_s"] * congestion_factor(d, hour)
    return _weight

# Choix d'une origine / destination si elles n'existent pas déjà
if "origin_node" not in globals() or "dest_node" not in globals():
    if "NUM_ROUTE" in gdf_work.columns and gdf_work["NUM_ROUTE"].notna().any():
        route_counts = gdf_work["NUM_ROUTE"].value_counts()
        eligible_routes = route_counts[route_counts >= 8].index.tolist()

        if len(eligible_routes) > 0:
            rng = np.random.default_rng(42)
            chosen_route = rng.choice(eligible_routes[:min(100, len(eligible_routes))])
            sub = gdf_work[gdf_work["NUM_ROUTE"] == chosen_route].sample(2, random_state=42)
        else:
            sub = gdf_work.sample(2, random_state=42)
    else:
        sub = gdf_work.sample(2, random_state=42)

    origin_geom = sub.iloc[0].geometry
    dest_geom = sub.iloc[1].geometry

    origin_point = origin_geom.interpolate(0.5, normalized=True)
    dest_point = dest_geom.interpolate(0.5, normalized=True)

    origin_node = nearest_node(origin_point.x, origin_point.y)
    dest_node = nearest_node(dest_point.x, dest_point.y)

print("Origine :", origin_node)
print("Destination :", dest_node)


Origine : (np.float64(563662.9), np.float64(6522773.2))
Destination : (np.float64(536199.2), np.float64(6688370.7))


In [8]:
# === Cellule 2 : comparaison multi-heures ===

hours_to_compare = [8, 12, 18, 22]
paths_by_hour = {}
results = []

for h in hours_to_compare:
    try:
        path = nx.shortest_path(
            G_base,
            source=origin_node,
            target=dest_node,
            weight=weight_for_hour(h)
        )

        dynamic_time_s = nx.shortest_path_length(
            G_base,
            source=origin_node,
            target=dest_node,
            weight=weight_for_hour(h)
        )

        base_time_s = sum(
            G_base[path[i]][path[i+1]]["travel_time_base_s"]
            for i in range(len(path) - 1)
        )

        dist_m = sum(
            G_base[path[i]][path[i+1]]["length_m"]
            for i in range(len(path) - 1)
        )

        avg_factor = dynamic_time_s / base_time_s if base_time_s > 0 else np.nan

        paths_by_hour[h] = path
        results.append({
            "heure": f"{h:02d}:00",
            "distance_km": round(dist_m / 1000, 2),
            "temps_base_min": round(base_time_s / 60, 2),
            "temps_congestion_min": round(dynamic_time_s / 60, 2),
            "surcout_congestion_%": round((avg_factor - 1) * 100, 1),
            "nb_noeuds": len(path),
            "status": "OK"
        })

    except nx.NetworkXNoPath:
        results.append({
            "heure": f"{h:02d}:00",
            "distance_km": np.nan,
            "temps_base_min": np.nan,
            "temps_congestion_min": np.nan,
            "surcout_congestion_%": np.nan,
            "nb_noeuds": np.nan,
            "status": "Aucun chemin"
        })

comparison_df = pd.DataFrame(results).sort_values("heure").reset_index(drop=True)
display(comparison_df)

valid_rows = comparison_df[comparison_df["status"] == "OK"].copy()
if len(valid_rows) > 0:
    best_hour = int(valid_rows.loc[valid_rows["temps_congestion_min"].idxmin(), "heure"][:2])
    worst_hour = int(valid_rows.loc[valid_rows["temps_congestion_min"].idxmax(), "heure"][:2])
    print(f"Heure la plus fluide : {best_hour:02d}:00")
    print(f"Heure la plus congestionnée : {worst_hour:02d}:00")
else:
    print("Aucune comparaison possible sur cet OD.")

,heure,distance_km,temps_base_min,temps_congestion_min,surcout_congestion_%,nb_noeuds,status
0,08:00,0.21,0.11,0.13,15.3,234,OK
1,12:00,0.21,0.11,0.11,-1.0,234,OK
2,18:00,0.21,0.11,0.13,15.3,234,OK
3,22:00,0.21,0.11,0.11,-5.3,234,OK


Heure la plus fluide : 12:00
Heure la plus congestionnée : 08:00


In [9]:
# === Cellule 3 : carte folium pour une heure donnée ===

HOUR_TO_MAP = 18  # change ici : 8, 12, 18, 22 ...

if HOUR_TO_MAP not in paths_by_hour:
    raise ValueError("Cette heure n'a pas encore été calculée dans la cellule précédente.")

path = paths_by_hour[HOUR_TO_MAP]

def extract_path_geometries(path, G):
    geoms = []
    for i in range(len(path) - 1):
        geom = G[path[i]][path[i+1]].get("geometry")
        if geom is not None:
            geoms.append(geom)
    return geoms

route_geoms = extract_path_geometries(path, G_base)

route_gs = gpd.GeoSeries(route_geoms, crs=gdf_work.crs).to_crs(4326)
od_points = gpd.GeoSeries([Point(origin_node), Point(dest_node)], crs=gdf_work.crs).to_crs(4326)

origin_ll = (od_points.iloc[0].y, od_points.iloc[0].x)
dest_ll = (od_points.iloc[1].y, od_points.iloc[1].x)

center_lat = (origin_ll[0] + dest_ll[0]) / 2
center_lon = (origin_ll[1] + dest_ll[1]) / 2

time_row = comparison_df[comparison_df["heure"] == f"{HOUR_TO_MAP:02d}:00"].iloc[0]

m = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles="CartoDB positron")

for geom in route_gs:
    if geom.geom_type == "LineString":
        coords = [(y, x) for x, y in geom.coords]
        folium.PolyLine(
            locations=coords,
            weight=5,
            opacity=0.85,
            tooltip=f"Itinéraire {HOUR_TO_MAP:02d}:00"
        ).add_to(m)

    elif geom.geom_type == "MultiLineString":
        for part in geom.geoms:
            coords = [(y, x) for x, y in part.coords]
            folium.PolyLine(
                locations=coords,
                weight=5,
                opacity=0.85,
                tooltip=f"Itinéraire {HOUR_TO_MAP:02d}:00"
            ).add_to(m)

folium.Marker(
    location=origin_ll,
    popup="Origine",
    tooltip="Origine"
).add_to(m)

folium.Marker(
    location=dest_ll,
    popup="Destination",
    tooltip="Destination"
).add_to(m)

title_html = f"""
<div style="
    position: fixed;
    top: 10px; left: 50px; width: 320px; z-index:9999;
    background-color: white; padding: 10px; border: 2px solid #444; border-radius: 8px;
    font-size: 14px;">
    <b>Itinéraire optimisé à {HOUR_TO_MAP:02d}:00</b><br>
    Distance : {time_row['distance_km']} km<br>
    Temps base : {time_row['temps_base_min']} min<br>
    Temps congestion : {time_row['temps_congestion_min']} min<br>
    Surcoût : {time_row['surcout_congestion_%']}%
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

m